# Generate Last-Layer Diagnostics Across Context Lengths

This Generate notebook computes and caches the data consumed by `Visualize-last-layer-diagnostics-across-context-lengths.ipynb`. The companion Visualize notebook illustrates how last-layer graph-spectral diagnostics vary across context lengths for the main numerical input families using Llama-3.2-3B, providing the final-layer comparison presented under **Robustness and Layerwise Consistency of Graph-Spectral Diagnostics** in the appendix.


Set imports, constants, output paths, sweep axes, and random seeds.


In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import random
from pathlib import Path

import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model and sweep configuration
SEED = 42
ENSEMBLE_SEED = SEED
MODEL_NAME = "meta-llama/Llama-3.2-3B"
EXPECTED_NUM_LAYERS = 28
NUM_ENSEMBLE_RUNS = 20
EXPECTED_NUM_INPUT_FAMILIES = 10

# Diagnostics computed at each context length
MAX_CONTEXT_STATES = 1000
TOTAL_SEQUENCE_STATES = MAX_CONTEXT_STATES
if TOTAL_SEQUENCE_STATES != MAX_CONTEXT_STATES:
    raise ValueError("This diagnostics workflow should generate exactly the prompt/context states.")

# Checkpoint progress; save final plot data to OUTPUT_PATH
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY_N_TASKS = 200

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = CACHE_DIR / "all_diagnostics_last_layer_vs_context_length.npz"
CHECKPOINT_PATH = CACHE_DIR / "all_diagnostics_last_layer_vs_context_length.resume_checkpoint.npz"

NPZ_SCHEMA_VERSION = "all_diagnostics_last_layer_vs_context_length_main_diagnostics"
CHECKPOINT_SCHEMA_VERSION = NPZ_SCHEMA_VERSION + "_checkpoint"

DROP_BOS = True
RENORMALIZE_AFTER_DROP = True
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
HEAD_WEIGHTING = "mass"
TOKENIZATION_ADD_SPECIAL_TOKENS = True
K_CUTOFF = None
LAST_LAYER_ATTENTION_LAYER_INDEX = -1
LAST_LAYER_SIGNAL_LAYER_INDEX = -2

# Sweep axes
context_fracs_full = np.asarray([i / 20 for i in range(1, 21)], dtype=np.float64)
context_percent_full = np.asarray([int(round(100 * frac)) for frac in context_fracs_full], dtype=np.int64)
run_ids_full = np.arange(NUM_ENSEMBLE_RUNS, dtype=np.int64)

input_data_order_full = np.asarray([
    "Constant sampled",
    "Logistic r=3.2",
    "Logistic r=3.5",
    "Logistic r=3.55",
    "Logistic r=3.566",
    "Logistic r=3.5692",
    "Logistic r=3.5698",
    "Logistic r=3.9",
    "Logistic r=4.0",
    "Lorenz",
], dtype=str)

# Run the full sweep
active_input_indices = np.arange(input_data_order_full.size, dtype=np.int64)
active_run_indices = np.arange(run_ids_full.size, dtype=np.int64)
active_context_indices = np.arange(context_fracs_full.size, dtype=np.int64)

input_names = input_data_order_full[active_input_indices]
run_ids = run_ids_full[active_run_indices]
context_fracs = context_fracs_full[active_context_indices]
context_percent = context_percent_full[active_context_indices]

diagnostic_names = np.asarray([
    "last_layer_effective_active_modes",
    "last_layer_normalized_fiedler_value_lambda_2",
    "last_layer_hfer",
], dtype=str)

# Seed data generation and model inference
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(
    f"Configured last-layer diagnostics sweep: {len(input_names)} inputs x {len(run_ids)} runs x "
    f"{len(context_fracs)} contexts."
)


Define sequence generators and labels for the input families.


In [ ]:
# Periodic and chaotic logistic-map regimes
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_expected_periods = {
    r: periodic_logistic_periods.get(r)
    for r in list(periodic_logistic_periods.keys()) + [3.9, 4.0]
}
logistic_long_n = 20000
logistic_settle_tol = 1e-10
periodic_random_phase = True

# Match Lorenz sampling to the sequence length
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
lorenz_dt = 100 / 999
t_eval = np.arange(TOTAL_SEQUENCE_STATES, dtype=float) * lorenz_dt
t_span = (float(t_eval[0]), float(t_eval[-1]))


def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


def logistic_sequence(r, x0, n):
    x = np.empty(int(n), dtype=float)
    x[0] = float(x0)
    for i in range(1, int(n)):
        x[i] = float(r) * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def build_burned_in_logistic_window_ensemble(
    r,
    x0,
    n_window,
    expected_period=None,
    n_long=20000,
    tol=1e-10,
    phase_offset=0,
):
    # Use settled periodic windows with an optional phase shift
    if expected_period is None:
        vals = logistic_sequence(r, x0, n_window)
        return np.asarray(vals, dtype=float), 0, 0, False

    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + n_window > x_long.size:
        raise ValueError(
            f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states."
        )

    vals = np.asarray(x_long[start_idx:start_idx + n_window], dtype=float)
    return vals, int(t_settle), int(phase_offset), True


def quantize_lorenz_x_values(x_values):
    x_values = np.asarray(x_values, dtype=float)
    x_min_rep = float(np.min(x_values))
    x_max_rep = float(np.max(x_values))
    if x_max_rep == x_min_rep:
        return np.full(x_values.shape[0], 500, dtype=int)
    return np.round(50 + (x_values - x_min_rep) / (x_max_rep - x_min_rep) * 900).astype(int)


def _display_name(data_name):
    display_names = {
        "Constant sampled": "Constant sampled",
        "Logistic r=3.2": "Logistic r=3.2 (period 2)",
        "Logistic r=3.5": "Logistic r=3.5 (period 4)",
        "Logistic r=3.55": "Logistic r=3.55 (period 8)",
        "Logistic r=3.566": "Logistic r=3.566 (period 16)",
        "Logistic r=3.5692": "Logistic r=3.5692 (period 32)",
        "Logistic r=3.5698": "Logistic r=3.5698 (period 64)",
        "Logistic r=3.9": "Logistic r=3.9 (chaotic)",
        "Logistic r=4.0": "Logistic r=4.0 (chaotic)",
        "Lorenz": "Lorenz",
    }
    return display_names.get(str(data_name), str(data_name))


def _input_class_label(data_name):
    data_name = str(data_name)
    if data_name == "Constant sampled":
        return "constant"
    if data_name in {f"Logistic r={r}" for r in periodic_logistic_periods}:
        return "periodic"
    if data_name in {"Logistic r=3.9", "Logistic r=4.0"}:
        return "chaotic"
    if data_name == "Lorenz":
        return "lorenz"
    raise ValueError(f"Unknown input family: {data_name}")


# Cache labels for visualization
input_labels = np.asarray([_display_name(name) for name in input_names], dtype=str)
input_class_labels = np.asarray([_input_class_label(name) for name in input_names], dtype=str)

if input_names.shape != (EXPECTED_NUM_INPUT_FAMILIES,):
    raise ValueError(f"Expected {EXPECTED_NUM_INPUT_FAMILIES} inputs, got {input_names.shape}.")
if run_ids.shape != (NUM_ENSEMBLE_RUNS,):
    raise ValueError(f"Expected {NUM_ENSEMBLE_RUNS} run ids, got {run_ids.shape}.")
if context_fracs.shape != (20,):
    raise ValueError(f"Expected 20 context fractions, got {context_fracs.shape}.")



Build the ensemble of constant, logistic, and Lorenz token sequences.


In [ ]:
# Share each replicate's x0 across input families
ensemble_rng = np.random.default_rng(ENSEMBLE_SEED)
x0_samples = ensemble_rng.uniform(0.3, 0.7, size=NUM_ENSEMBLE_RUNS)
phase_rng = np.random.default_rng(ENSEMBLE_SEED + 1)

ensemble_periodic_rs = [3.2, 3.5, 3.55, 3.566, 3.5692, 3.5698]
ensemble_chaotic_rs = [3.9, 4.0]

constant_tokens = np.full(NUM_ENSEMBLE_RUNS, -1, dtype=np.int64)
ensemble_inputs = {}

for rep_id, x0_rep in enumerate(x0_samples):
    rep_id = int(rep_id)
    x0_rep = float(x0_rep)
    rep_inputs = {}

    lorenz_x0_rep = float(-8.0 + (x0_rep - 0.5))
    lorenz_y0_rep = 7.0
    lorenz_z0_rep = 27.0

    # Repeat one token per constant replicate
    constant_token = int(np.round(50 + 900 * x0_rep))
    constant_tokens[rep_id] = constant_token
    rep_inputs["Constant sampled"] = {
        "sequence": np.full(TOTAL_SEQUENCE_STATES, constant_token, dtype=int),
        "metadata": {
            "input_class": "constant",
            "rep_id": rep_id,
            "x0_rep": x0_rep,
            "constant_token": constant_token,
        },
    }

    # Use settled periodic windows with randomized phase
    for r in ensemble_periodic_rs:
        expected_period = logistic_expected_periods[r]
        if expected_period is None:
            raise ValueError(f"Expected a periodic window for r={r}, but expected_period is None.")
        phase_offset = (
            int(phase_rng.integers(0, expected_period))
            if periodic_random_phase
            else 0
        )
        vals, burn_in_start_index, used_phase_offset, used_burn_in = build_burned_in_logistic_window_ensemble(
            r,
            x0=x0_rep,
            n_window=TOTAL_SEQUENCE_STATES,
            expected_period=expected_period,
            n_long=logistic_long_n,
            tol=logistic_settle_tol,
            phase_offset=phase_offset,
        )
        data_name = f"Logistic r={r}"
        rep_inputs[data_name] = {
            "sequence": quantize_logistic_window(vals),
            "metadata": {
                "input_class": "periodic",
                "rep_id": rep_id,
                "x0_rep": x0_rep,
                "r": float(r),
                "expected_period": int(expected_period),
                "burn_in_start_index": int(burn_in_start_index),
                "phase_offset": int(used_phase_offset),
                "used_burn_in": bool(used_burn_in),
            },
        }

    # Start chaotic trajectories from the sampled x0
    for r in ensemble_chaotic_rs:
        vals = logistic_sequence(r, x0_rep, TOTAL_SEQUENCE_STATES)
        data_name = f"Logistic r={r}"
        rep_inputs[data_name] = {
            "sequence": quantize_logistic_window(vals),
            "metadata": {
                "input_class": "chaotic",
                "rep_id": rep_id,
                "x0_rep": x0_rep,
                "r": float(r),
                "burn_in_start_index": 0,
                "phase_offset": 0,
                "used_burn_in": False,
            },
        }

    # Perturb Lorenz x0 and quantize the x-coordinate
    sol_rep = solve_ivp(
        lorenz,
        t_span,
        [lorenz_x0_rep, lorenz_y0_rep, lorenz_z0_rep],
        args=(sigma, rho, beta),
        t_eval=t_eval,
    )
    if not sol_rep.success:
        raise RuntimeError(f"Lorenz solve failed for rep_id={rep_id}: {sol_rep.message}")
    if sol_rep.y.shape[-1] != TOTAL_SEQUENCE_STATES:
        raise ValueError(
            f"Expected {TOTAL_SEQUENCE_STATES} Lorenz samples for rep_id={rep_id}, got {sol_rep.y.shape[-1]}."
        )
    rep_inputs["Lorenz"] = {
        "sequence": quantize_lorenz_x_values(sol_rep.y[0]),
        "metadata": {
            "input_class": "lorenz",
            "rep_id": rep_id,
            "x0_rep": x0_rep,
            "lorenz_x0_rep": lorenz_x0_rep,
            "lorenz_y0_rep": lorenz_y0_rep,
            "lorenz_z0_rep": lorenz_z0_rep,
        },
    }

    if list(rep_inputs.keys()) != input_data_order_full.tolist():
        raise ValueError(f"rep_id={rep_id}: unexpected input order {list(rep_inputs.keys())}.")

    # Validate token sequences before model inference
    for data_name in input_data_order_full:
        sequence = np.asarray(rep_inputs[str(data_name)]["sequence"], dtype=int)
        if sequence.shape != (TOTAL_SEQUENCE_STATES,):
            raise ValueError(
                f"{data_name}, rep_id={rep_id}: expected {TOTAL_SEQUENCE_STATES} states, got {sequence.shape}."
            )
        if not np.all((0 <= sequence) & (sequence <= 999)):
            raise ValueError(f"{data_name}, rep_id={rep_id}: quantized states must be in [0, 999].")

    ensemble_inputs[rep_id] = rep_inputs

for rep_id in run_ids:
    for data_name in input_names:
        if int(rep_id) not in ensemble_inputs or str(data_name) not in ensemble_inputs[int(rep_id)]:
            raise ValueError(f"Missing ensemble input for rep_id={int(rep_id)}, data_name={str(data_name)}.")

if len(ensemble_inputs) != NUM_ENSEMBLE_RUNS:
    raise ValueError(f"Expected {NUM_ENSEMBLE_RUNS} ensemble runs, got {len(ensemble_inputs)}.")
if len(set(constant_tokens.tolist())) <= 1:
    raise ValueError("Expected sampled constant tokens to vary across ensemble replicates.")

print(f"Built ensemble inputs: {len(ensemble_inputs)} runs x {len(input_names)} inputs.")


Load the tokenizer and validate prompt tokenization.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def _single_token_id(token_str):
    token_ids = tokenizer(token_str, add_special_tokens=False).input_ids
    if len(token_ids) != 1:
        raise ValueError(f"Expected {token_str!r} to tokenize as one token; got {token_ids}.")
    return int(token_ids[0])


COMMA_TOKEN_ID = _single_token_id(",")
# Require one token per state and comma for exact context sizing
NUMERIC_TOKEN_IDS = np.asarray([_single_token_id(f"{i:03d}") for i in range(1000)], dtype=np.int64)
if len(np.unique(NUMERIC_TOKEN_IDS)) != 1000:
    raise ValueError("Expected all 000..999 formatted states to map to distinct single tokens.")


def _first_token_value(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        if x.numel() == 0:
            return None
        return int(x.flatten()[0].item())
    arr = np.asarray(x).reshape(-1)
    if arr.size == 0:
        return None
    return int(arr[0])


def get_token_start_index(
    input_ids=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
):
    # Exclude an optional leading BOS from graph and signal nodes
    if not drop_bos:
        return 0

    first_id = _first_token_value(input_ids)
    if first_id is not None and bos_token_id is not None and first_id == int(bos_token_id):
        return 1

    first_mask = _first_token_value(special_tokens_mask)
    if first_mask == 1:
        return 1

    return 0


def _build_input_sequence_from_frac(sequence, frac, total_n=MAX_CONTEXT_STATES):
    # Build comma-delimited prompts at each context fraction
    sequence = np.asarray(sequence, dtype=int)
    if sequence.size < int(total_n):
        raise ValueError(f"Need at least {int(total_n)} states; got {sequence.size}.")
    if np.min(sequence[:int(total_n)]) < 0 or np.max(sequence[:int(total_n)]) > 999:
        raise ValueError("Prompt state tokens must be in [0, 999].")

    frac_use = float(frac)
    if frac_use >= 1.0:
        ctx_len = int(total_n)
    else:
        ctx_len = max(1, int(round(frac_use * int(total_n))))
    ctx_len = min(int(total_n), ctx_len)
    context_window = sequence[:ctx_len]
    input_sequence = ",".join([f"{x:03d}" for x in context_window]) + ","
    return input_sequence, int(ctx_len)


def _post_bos_token_ids(input_ids, token_start):
    flat_ids = input_ids.reshape(-1)
    return flat_ids[int(token_start):]


def tokenize_and_validate_diagnostic_prompt(prompt, expected_post_bos_tokens=None):
    if not prompt.endswith(","):
        raise ValueError("Diagnostic prompt must end with a comma.")

    encoded = tokenizer(
        prompt,
        add_special_tokens=TOKENIZATION_ADD_SPECIAL_TOKENS,
        return_special_tokens_mask=True,
        return_tensors="pt",
    )
    input_ids = encoded.input_ids[0]
    special_tokens_mask = encoded.special_tokens_mask[0]
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=tokenizer.bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=DROP_BOS,
    )
    post_bos_ids = _post_bos_token_ids(input_ids, token_start)
    seq_len = int(post_bos_ids.shape[0])
    if expected_post_bos_tokens is not None and seq_len != int(expected_post_bos_tokens):
        raise ValueError(
            f"Expected {int(expected_post_bos_tokens)} post-BOS content tokens, got {seq_len}."
        )
    if seq_len <= 1:
        raise ValueError(f"Need at least 2 post-BOS tokens for lambda_2; got {seq_len}.")
    if int(post_bos_ids[-1].item()) != COMMA_TOKEN_ID:
        decoded_final = tokenizer.decode([int(post_bos_ids[-1].item())], clean_up_tokenization_spaces=False)
        raise ValueError(f"Final post-BOS token is not comma; got {decoded_final!r}.")
    return {
        "encoded": encoded,
        "token_start": int(token_start),
        "seq_len": int(seq_len),
        "final_token_is_comma": True,
    }


reference_sequence = np.arange(MAX_CONTEXT_STATES, dtype=int) % 1000
context_state_counts = np.zeros(len(context_fracs), dtype=np.int64)
context_token_counts = np.zeros(len(context_fracs), dtype=np.int64)
# Measure exact post-BOS lengths with a reference prompt
for context_idx, frac in enumerate(context_fracs):
    reference_prompt, ctx_len = _build_input_sequence_from_frac(
        reference_sequence,
        frac,
        total_n=MAX_CONTEXT_STATES,
    )
    prompt_info = tokenize_and_validate_diagnostic_prompt(reference_prompt)
    context_state_counts[context_idx] = int(ctx_len)
    context_token_counts[context_idx] = int(prompt_info["seq_len"])

expected_context_state_counts = np.maximum(
    1,
    np.rint(context_fracs * MAX_CONTEXT_STATES).astype(np.int64),
)
if not np.array_equal(context_state_counts, expected_context_state_counts):
    raise ValueError(
        f"Expected exact context state counts {expected_context_state_counts.tolist()}, "
        f"got {context_state_counts.tolist()}."
    )
if not np.array_equal(context_token_counts, 2 * context_state_counts):
    raise ValueError("Expected one numeric token and one comma token per context state.")
if not np.all(np.diff(context_token_counts) > 0):
    raise ValueError(f"context_token_counts must be strictly increasing; got {context_token_counts.tolist()}.")
if int(context_token_counts[-1]) != 2000:
    raise ValueError(f"Full context must be 2000 post-BOS content tokens; got {context_token_counts[-1]}.")
if K_CUTOFF is not None:
    k_cutoff_value = int(K_CUTOFF)
    if k_cutoff_value < 0 or np.any(k_cutoff_value > context_token_counts):
        raise ValueError("K_CUTOFF must be in [0, seq_len] for every context length.")

print(
    f"Tokenizer ready: {len(context_token_counts)} contexts, "
    f"{int(context_token_counts[0])}..{int(context_token_counts[-1])} post-BOS tokens."
)


Compute the graph-spectral diagnostics from the configured last-layer attention and hidden-state tensors.


In [ ]:
def _resolve_attention_layer_index(attentions, layer_index):
    resolved_layer_index = layer_index if layer_index >= 0 else len(attentions) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(attentions):
        raise ValueError(
            f"Invalid attention layer_index={layer_index} for {len(attentions)} attention tensors."
        )
    return int(resolved_layer_index)


def _resolve_hidden_state_layer_index(hidden_states, layer_index):
    resolved_layer_index = layer_index if layer_index >= 0 else len(hidden_states) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(hidden_states):
        raise ValueError(
            f"Invalid hidden-state layer_index={layer_index} for {len(hidden_states)} hidden-state tensors."
        )
    return int(resolved_layer_index)


def compute_last_layer_diagnostics_from_outputs(
    attentions,
    hidden_states,
    input_ids,
    expected_seq_len,
    k_cutoff=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
    renormalize_after_drop=True,
    head_weighting="mass",
    attention_layer_index=LAST_LAYER_ATTENTION_LAYER_INDEX,
    signal_layer_index=LAST_LAYER_SIGNAL_LAYER_INDEX,
):
    # Remove the same leading BOS from attention and hidden states
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=drop_bos,
    )
    # Resolve the configured last-layer attention/signal tensors before graph construction
    resolved_attention_layer_index = _resolve_attention_layer_index(attentions, attention_layer_index)
    resolved_signal_layer_index = _resolve_hidden_state_layer_index(hidden_states, signal_layer_index)

    A = attentions[resolved_attention_layer_index][0].float()
    A = A[:, token_start:, token_start:]
    seq_len = int(A.shape[-1])
    if seq_len != int(expected_seq_len):
        raise ValueError(f"Expected seq_len={expected_seq_len}, got {seq_len}.")
    if seq_len <= 1:
        raise ValueError(f"seq_len={seq_len} is too small for lambda_2.")

    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    # Symmetrize heads and combine them into one weighted graph
    if k_cutoff is None:
        k_cutoff = seq_len // 2
    k_cutoff = int(k_cutoff)
    if k_cutoff < 0 or k_cutoff > seq_len:
        raise ValueError(f"Invalid k_cutoff={k_cutoff} for seq_len={seq_len}.")

    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == "uniform":
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == "mass":
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'.")

    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    deg = Wbar.sum(dim=1)

    L = torch.diag(deg) - Wbar
    _, U = torch.linalg.eigh(L)

    # Project hidden states onto graph Fourier modes
    X = hidden_states[resolved_signal_layer_index][0].float()
    X = X[token_start:, :].to(U.device)
    if X.shape[0] != seq_len:
        raise ValueError(
            f"Signal/token length mismatch after BOS handling: X has {X.shape[0]} rows, graph has {seq_len} nodes."
        )

    X_hat = U.T @ X
    energy = (X_hat ** 2).sum(dim=1)
    total_energy = energy.sum()
    if total_energy <= 0 or not torch.isfinite(total_energy):
        raise ValueError(f"Nonpositive or nonfinite total signal energy: {total_energy}.")
    p = energy / total_energy
    positive_p = p[p > 0]
    spectral_entropy = -torch.sum(positive_p * torch.log(positive_p))
    effective_active_modes = torch.exp(spectral_entropy)
    hfer = float(p[k_cutoff:].sum().detach().cpu().item())

    inv_sqrt_deg = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
    Wbar_norm = inv_sqrt_deg[:, None] * Wbar * inv_sqrt_deg[None, :]
    L_sym = torch.eye(seq_len, device=Wbar.device, dtype=Wbar.dtype) - Wbar_norm
    eigvals_sym = torch.linalg.eigvalsh(L_sym)
    normalized_fiedler_value = float(eigvals_sym[1].detach().cpu().item())

    result = {
        "effective_active_modes": float(effective_active_modes.detach().cpu().item()),
        "normalized_fiedler_value": normalized_fiedler_value,
        "hfer": hfer,
        "seq_len": int(seq_len),
        "k_cutoff": int(k_cutoff),
        "token_start": int(token_start),
        "attention_layer_index": int(resolved_attention_layer_index),
        "signal_layer_index": int(resolved_signal_layer_index),
    }

    del A, W_heads, Wbar, deg, L, U, X, X_hat, energy, p, inv_sqrt_deg, Wbar_norm, L_sym, eigvals_sym
    return result


Create, validate, load, and save checkpoint arrays for resumable sweeps.


In [ ]:
def _expected_result_shapes():
    return (len(input_names), len(run_ids), len(context_fracs))


def _initialize_result_arrays():
    # Retain per-run values for summary recomputation
    base_shape = _expected_result_shapes()
    return {
        "active_modes_last_layer_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "fiedler_last_layer_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "hfer_last_layer_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "completed_mask": np.zeros(base_shape, dtype=bool),
    }


def _npz_scalar_to_str(data, key):
    return str(np.asarray(data[key]).item())


def _validate_checkpoint_compatibility(data, num_layers):
    # Accept only checkpoints matching the model, sweep, and schema
    if _npz_scalar_to_str(data, "checkpoint_schema_version") != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError("Checkpoint schema version mismatch.")
    if _npz_scalar_to_str(data, "npz_schema_version") != NPZ_SCHEMA_VERSION:
        raise ValueError("Checkpoint final schema version mismatch.")
    if _npz_scalar_to_str(data, "model_name") != MODEL_NAME:
        raise ValueError("Checkpoint model_name mismatch.")
    if int(np.asarray(data["num_layers"]).item()) != int(num_layers):
        raise ValueError("Checkpoint num_layers mismatch.")
    if int(np.asarray(data["last_layer_attention_layer_index"]).item()) != int(LAST_LAYER_ATTENTION_LAYER_INDEX):
        raise ValueError("Checkpoint attention layer-index convention mismatch.")
    if int(np.asarray(data["last_layer_signal_layer_index"]).item()) != int(LAST_LAYER_SIGNAL_LAYER_INDEX):
        raise ValueError("Checkpoint signal layer-index convention mismatch.")
    if not np.array_equal(data["input_names"].astype(str), input_names.astype(str)):
        raise ValueError("Checkpoint input_names mismatch.")
    if not np.array_equal(data["run_ids"].astype(np.int64), run_ids.astype(np.int64)):
        raise ValueError("Checkpoint run_ids mismatch.")
    if not np.allclose(data["context_fracs"].astype(np.float64), context_fracs.astype(np.float64)):
        raise ValueError("Checkpoint context_fracs mismatch.")
    if int(np.asarray(data["max_context_states"]).item()) != MAX_CONTEXT_STATES:
        raise ValueError("Checkpoint max_context_states mismatch.")
    if int(np.asarray(data["total_sequence_states"]).item()) != TOTAL_SEQUENCE_STATES:
        raise ValueError("Checkpoint total_sequence_states mismatch.")
    if not np.array_equal(data["context_state_counts"].astype(np.int64), context_state_counts.astype(np.int64)):
        raise ValueError("Checkpoint context_state_counts mismatch.")
    if not np.array_equal(data["context_token_counts"].astype(np.int64), context_token_counts.astype(np.int64)):
        raise ValueError("Checkpoint context_token_counts mismatch.")

    expected_base_shape = _expected_result_shapes()
    shape_checks = {
        "active_modes_last_layer_per_run": expected_base_shape,
        "fiedler_last_layer_per_run": expected_base_shape,
        "hfer_last_layer_per_run": expected_base_shape,
        "completed_mask": expected_base_shape,
    }
    for key, expected_shape in shape_checks.items():
        if data[key].shape != expected_shape:
            raise ValueError(f"Checkpoint {key} shape {data[key].shape}, expected {expected_shape}.")


def load_or_initialize_checkpoint(num_layers):
    if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
        print(f"Loading checkpoint from {CHECKPOINT_PATH}")
        with np.load(CHECKPOINT_PATH, allow_pickle=False) as data:
            _validate_checkpoint_compatibility(data, num_layers=num_layers)
            arrays = {
                "active_modes_last_layer_per_run": data["active_modes_last_layer_per_run"].astype(np.float64),
                "fiedler_last_layer_per_run": data["fiedler_last_layer_per_run"].astype(np.float64),
                "hfer_last_layer_per_run": data["hfer_last_layer_per_run"].astype(np.float64),
                "completed_mask": data["completed_mask"].astype(bool),
            }
        print(f"  completed tasks in checkpoint: {int(arrays['completed_mask'].sum())}")
        return arrays

    return _initialize_result_arrays()


def save_checkpoint(arrays, num_layers):
    # Save progress with compatibility metadata
    np.savez_compressed(
        CHECKPOINT_PATH,
        input_names=input_names,
        context_fracs=context_fracs,
        context_state_counts=context_state_counts,
        context_token_counts=context_token_counts,
        run_ids=run_ids,
        model_name=np.asarray(MODEL_NAME),
        max_context_states=np.asarray(MAX_CONTEXT_STATES, dtype=np.int64),
        total_sequence_states=np.asarray(TOTAL_SEQUENCE_STATES, dtype=np.int64),
        num_layers=np.asarray(num_layers, dtype=np.int64),
        last_layer_attention_layer_index=np.asarray(LAST_LAYER_ATTENTION_LAYER_INDEX, dtype=np.int64),
        last_layer_signal_layer_index=np.asarray(LAST_LAYER_SIGNAL_LAYER_INDEX, dtype=np.int64),
        npz_schema_version=np.asarray(NPZ_SCHEMA_VERSION),
        checkpoint_schema_version=np.asarray(CHECKPOINT_SCHEMA_VERSION),
        active_modes_last_layer_per_run=arrays["active_modes_last_layer_per_run"],
        fiedler_last_layer_per_run=arrays["fiedler_last_layer_per_run"],
        hfer_last_layer_per_run=arrays["hfer_last_layer_per_run"],
        completed_mask=arrays["completed_mask"],
    )
    completed_total = int(arrays["completed_mask"].sum())
    total_tasks = int(arrays["completed_mask"].size)
    completed_percent = 100.0 * completed_total / total_tasks
    print(
        f"Checkpoint saved: {completed_total}/{total_tasks} tasks complete "
        f"({completed_percent:.1f}%)"
    )


Load the model with eager attention and capture layer/device metadata.


In [ ]:
# Use eager attention to return full attention tensors
print("Loading model...")
model_attn = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model_attn.eval()

num_layers = int(model_attn.config.num_hidden_layers)
if num_layers != EXPECTED_NUM_LAYERS:
    raise ValueError(f"Expected model to have {EXPECTED_NUM_LAYERS} layers, got {num_layers}.")

layer_indices = np.arange(num_layers, dtype=np.int64)
layer_numbers = np.arange(1, num_layers + 1, dtype=np.int64)
if not np.array_equal(layer_numbers, np.arange(1, EXPECTED_NUM_LAYERS + 1, dtype=np.int64)):
    raise ValueError("Expected layer_numbers == 1..28.")

# Resolve last-layer conventions against the model output tuple lengths
resolved_last_attention_layer_index = _resolve_attention_layer_index(
    [None] * num_layers,
    LAST_LAYER_ATTENTION_LAYER_INDEX,
)
resolved_last_signal_layer_index = _resolve_hidden_state_layer_index(
    [None] * (num_layers + 1),
    LAST_LAYER_SIGNAL_LAYER_INDEX,
)
if resolved_last_attention_layer_index != num_layers - 1:
    raise ValueError("Last-layer attention convention must resolve to the final attention layer.")
if resolved_last_signal_layer_index != num_layers - 1:
    raise ValueError("Last-layer signal convention must resolve to hidden_states[-2], the pre-final-block signal.")

device = next(model_attn.parameters()).device
print(f"Model loaded: {MODEL_NAME}, {num_layers} layers.")
print(
    "Last-layer conventions: "
    f"attention index {LAST_LAYER_ATTENTION_LAYER_INDEX} -> {resolved_last_attention_layer_index}; "
    f"signal index {LAST_LAYER_SIGNAL_LAYER_INDEX} -> {resolved_last_signal_layer_index}."
)


Run or resume the full sweep, storing diagnostics from the configured last layer and checkpointing progress.


In [ ]:
arrays = load_or_initialize_checkpoint(num_layers=num_layers)
completed_since_checkpoint = 0

# Task grid: input family x realization x context length
tasks = [
    (input_axis_idx, str(data_name), run_axis_idx, int(rep_id), context_idx)
    for input_axis_idx, data_name in enumerate(input_names)
    for run_axis_idx, rep_id in enumerate(run_ids)
    for context_idx in range(len(context_fracs))
]

checkpoint_every = max(1, int(CHECKPOINT_EVERY_N_TASKS))
initial_completed_total = int(arrays["completed_mask"].sum())
next_checkpoint_completed = (
    (initial_completed_total // checkpoint_every) + 1
) * checkpoint_every

print(f"Running last-layer diagnostics sweep: {len(tasks)} tasks; checkpoint every {checkpoint_every} completed tasks.")

for input_axis_idx, data_name, run_axis_idx, rep_id, context_idx in tqdm(
    tasks,
    desc="All diagnostics last-layer vs context",
    miniters=checkpoint_every,
):
    # Skip tasks saved in a compatible checkpoint
    if arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx]:
        continue

    sequence = np.asarray(ensemble_inputs[int(rep_id)][data_name]["sequence"], dtype=int)
    prompt, ctx_len = _build_input_sequence_from_frac(
        sequence,
        context_fracs[context_idx],
        total_n=MAX_CONTEXT_STATES,
    )
    if int(ctx_len) != int(context_state_counts[context_idx]):
        raise ValueError(
            f"Context-state mismatch for {data_name}, rep_id={rep_id}, context_idx={context_idx}: "
            f"builder returned {ctx_len}, expected {context_state_counts[context_idx]}."
        )

    prompt_info = tokenize_and_validate_diagnostic_prompt(
        prompt,
        expected_post_bos_tokens=int(context_token_counts[context_idx]),
    )
    if not bool(prompt_info["final_token_is_comma"]):
        raise ValueError("Every prompt must end with the comma token.")
    encoded = prompt_info["encoded"]
    model_inputs = {
        key: value.to(device)
        for key, value in encoded.items()
        if key in {"input_ids", "attention_mask", "position_ids", "token_type_ids"}
    }
    input_ids_for_diagnostics = model_inputs["input_ids"][0]
    special_tokens_mask = (
        encoded["special_tokens_mask"][0].to(device)
        if "special_tokens_mask" in encoded
        else None
    )
    expected_seq_len = int(context_token_counts[context_idx])
    expected_k_cutoff = expected_seq_len // 2 if K_CUTOFF is None else int(K_CUTOFF)

    # Request attention and hidden states for the last layer
    with torch.no_grad():
        outputs = model_attn(
            **model_inputs,
            output_attentions=True,
            output_hidden_states=True,
            return_dict=True,
        )

    if outputs.attentions is None or outputs.hidden_states is None:
        raise ValueError("Model outputs must include attentions and hidden_states.")
    if len(outputs.attentions) != num_layers:
        raise ValueError(f"Expected {num_layers} attention tensors, got {len(outputs.attentions)}.")
    if len(outputs.hidden_states) != num_layers + 1:
        raise ValueError(f"Expected {num_layers + 1} hidden-state tensors, got {len(outputs.hidden_states)}.")

    # Compute each diagnostic from the configured last-layer tensors
    bundle = compute_last_layer_diagnostics_from_outputs(
        outputs.attentions,
        outputs.hidden_states,
        input_ids=input_ids_for_diagnostics,
        expected_seq_len=expected_seq_len,
        k_cutoff=K_CUTOFF,
        bos_token_id=tokenizer.bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=DROP_BOS,
        renormalize_after_drop=RENORMALIZE_AFTER_DROP,
        head_weighting=HEAD_WEIGHTING,
        attention_layer_index=LAST_LAYER_ATTENTION_LAYER_INDEX,
        signal_layer_index=LAST_LAYER_SIGNAL_LAYER_INDEX,
    )
    if int(bundle["attention_layer_index"]) != resolved_last_attention_layer_index:
        raise ValueError("Attention layer index does not match the last-layer convention.")
    if int(bundle["signal_layer_index"]) != resolved_last_signal_layer_index:
        raise ValueError("Signal layer index does not match the last-layer convention.")
    if int(bundle["seq_len"]) != expected_seq_len:
        raise ValueError("Graph seq_len must equal the saved context_token_counts value.")
    if int(bundle["k_cutoff"]) != expected_k_cutoff:
        raise ValueError("HFER k_cutoff must match the configured half-spectrum split.")

    arrays["active_modes_last_layer_per_run"][input_axis_idx, run_axis_idx, context_idx] = float(
        bundle["effective_active_modes"]
    )
    arrays["fiedler_last_layer_per_run"][input_axis_idx, run_axis_idx, context_idx] = float(
        bundle["normalized_fiedler_value"]
    )
    arrays["hfer_last_layer_per_run"][input_axis_idx, run_axis_idx, context_idx] = float(bundle["hfer"])
    arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx] = True
    completed_since_checkpoint += 1

    # Release large tensors between tasks
    del outputs, model_inputs, encoded, prompt_info, input_ids_for_diagnostics, special_tokens_mask, bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    completed_total = int(arrays["completed_mask"].sum())
    if completed_total >= next_checkpoint_completed:
        save_checkpoint(arrays, num_layers=num_layers)
        completed_since_checkpoint = 0
        next_checkpoint_completed = (
            (completed_total // checkpoint_every) + 1
        ) * checkpoint_every

if completed_since_checkpoint > 0 or not CHECKPOINT_PATH.exists():
    save_checkpoint(arrays, num_layers=num_layers)

print(f"Sweep complete: {int(arrays['completed_mask'].sum())}/{arrays['completed_mask'].size} tasks.")


Validate completed arrays, summarize over runs, and save the visualization cache.


In [ ]:
def _summarize_over_runs(per_run):
    mean = np.mean(per_run, axis=1)
    if per_run.shape[1] > 1:
        std = np.std(per_run, axis=1, ddof=1)
    else:
        std = np.zeros_like(mean)
    sem = std / np.sqrt(per_run.shape[1])
    return mean, std, sem


def _validate_final_arrays():
    # Reject incomplete or invalid arrays before saving
    if not np.all(arrays["completed_mask"]):
        bad = np.argwhere(~arrays["completed_mask"])[:10]
        raise ValueError(f"Not all tasks are complete; first incomplete indices: {bad}.")

    base_shape = _expected_result_shapes()
    expected_shapes = {
        "active_modes_last_layer_per_run": base_shape,
        "fiedler_last_layer_per_run": base_shape,
        "hfer_last_layer_per_run": base_shape,
    }
    expected_shapes.update({
        "active_modes_last_layer_per_run": (10, 20, 20),
        "fiedler_last_layer_per_run": (10, 20, 20),
        "hfer_last_layer_per_run": (10, 20, 20),
    })
    for key, expected_shape in expected_shapes.items():
        if arrays[key].shape != expected_shape:
            raise ValueError(f"{key} shape {arrays[key].shape}, expected {expected_shape}.")

    active = arrays["active_modes_last_layer_per_run"]
    fiedler = arrays["fiedler_last_layer_per_run"]
    hfer = arrays["hfer_last_layer_per_run"]

    for key, value in {
        "active_modes_last_layer_per_run": active,
        "fiedler_last_layer_per_run": fiedler,
        "hfer_last_layer_per_run": hfer,
    }.items():
        if not np.isfinite(value).all():
            bad = np.argwhere(~np.isfinite(value))[:10]
            raise ValueError(f"{key} contains nonfinite values; first bad indices: {bad}.")

    if not np.all(active > 0):
        raise ValueError("Active modes must be positive.")
    if not np.all((hfer >= -1e-6) & (hfer <= 1.0 + 1e-6)):
        raise ValueError("HFER values must be in [0, 1] up to tiny numerical tolerance.")
    if not np.all((fiedler >= -1e-4) & (fiedler <= 2.0 + 1e-4)):
        raise ValueError("Normalized Fiedler values must be in [0, 2] up to numerical tolerance.")

    if not np.all(np.diff(context_token_counts) > 0):
        raise ValueError("context_token_counts must be strictly increasing.")
    if int(context_token_counts[-1]) != 2000:
        raise ValueError("Final context length must be 2000 post-BOS content tokens.")


_validate_final_arrays()

# Summarize across realizations
active_modes_mean, active_modes_std, active_modes_sem = _summarize_over_runs(
    arrays["active_modes_last_layer_per_run"]
)
fiedler_mean, fiedler_std, fiedler_sem = _summarize_over_runs(
    arrays["fiedler_last_layer_per_run"]
)
hfer_mean, hfer_std, hfer_sem = _summarize_over_runs(
    arrays["hfer_last_layer_per_run"]
)

expected_summary_shape = (len(input_names), len(context_fracs))
for key, value, expected_shape in [
    ("active_modes_mean", active_modes_mean, expected_summary_shape),
    ("active_modes_std", active_modes_std, expected_summary_shape),
    ("active_modes_sem", active_modes_sem, expected_summary_shape),
    ("fiedler_mean", fiedler_mean, expected_summary_shape),
    ("fiedler_std", fiedler_std, expected_summary_shape),
    ("fiedler_sem", fiedler_sem, expected_summary_shape),
    ("hfer_mean", hfer_mean, expected_summary_shape),
    ("hfer_std", hfer_std, expected_summary_shape),
    ("hfer_sem", hfer_sem, expected_summary_shape),
]:
    if value.shape != expected_shape:
        raise ValueError(f"{key} shape {value.shape}, expected {expected_shape}.")
    if not np.isfinite(value).all():
        raise ValueError(f"{key} contains nonfinite values.")

# Save arrays for visualization
np.savez_compressed(
    OUTPUT_PATH,
    input_names=input_names,
    context_token_counts=context_token_counts,
    run_ids=run_ids,
    num_layers=np.asarray(num_layers, dtype=np.int64),
    num_ensemble_runs=np.asarray(len(run_ids), dtype=np.int64),
    model_name=np.asarray(MODEL_NAME),
    npz_schema_version=np.asarray(NPZ_SCHEMA_VERSION),
    diagnostic_names=diagnostic_names,
    diagnostic_layer_mode=np.asarray("last_layer"),
    last_layer_attention_layer_index=np.asarray(LAST_LAYER_ATTENTION_LAYER_INDEX, dtype=np.int64),
    last_layer_signal_layer_index=np.asarray(LAST_LAYER_SIGNAL_LAYER_INDEX, dtype=np.int64),
    last_layer_resolved_attention_layer_index=np.asarray(resolved_last_attention_layer_index, dtype=np.int64),
    last_layer_resolved_signal_layer_index=np.asarray(resolved_last_signal_layer_index, dtype=np.int64),
    active_modes_mean=active_modes_mean,
    active_modes_std=active_modes_std,
    fiedler_mean=fiedler_mean,
    fiedler_std=fiedler_std,
    hfer_mean=hfer_mean,
    hfer_std=hfer_std,
    active_modes_last_layer_per_run=arrays["active_modes_last_layer_per_run"],
    fiedler_last_layer_per_run=arrays["fiedler_last_layer_per_run"],
    hfer_last_layer_per_run=arrays["hfer_last_layer_per_run"],
)
